# Lab 2 — Enter the LLM: Same Task, Completely Different Machine

---

> **Lessons covered**: 02 Predictive vs Generative Systems · 03 Discriminative vs Generative Models  
> **Metric used**: Accuracy (same as Lab 1 — on purpose)

---

## Learning Goal

1. See what an LLM actually returns when you ask it a yes/no question
2. Experience the engineering challenge of turning free text into a usable label
3. Understand why `response_format` exists and when to use it
4. Compare LLM accuracy to XGBoost from Lab 1
5. See the same LLM behave as a **generative** system with a single prompt change

**Instructions**: Fill in the `# TODO` cells. Run them in order. If stuck, check `solution.ipynb`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

from openai import OpenAI
import json
import time

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

client = OpenAI()
print('Imports OK. OpenAI client ready.')

Imports OK. OpenAI client ready.


In [3]:
df = pd.read_csv(Path('../data/loan_applications.csv'))

FEATURES = [
    'credit_score', 'annual_income', 'loan_amount',
    'num_defaults', 'employment_years', 'age', 'loan_purpose'
]
TARGET = 'approved'

le = LabelEncoder()
df['loan_purpose_enc'] = le.fit_transform(df['loan_purpose'])
FEATURES_ENC = FEATURES[:-1] + ['loan_purpose_enc']

X = df[FEATURES_ENC]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Dataset: {len(df)} applicants | Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Approval rate: {y.mean():.1%}')

Dataset: 1000 applicants | Train: 800 | Test: 200
Approval rate: 65.0%


---
## Section 1 — The Naive Attempt: Just Ask It

In Lab 1 you called `model.predict(X_test)` and got back a numpy array of 0s and 1s.
Let's try the same thing with an LLM — just ask it the question directly.

In [4]:
# Pick one applicant from the test set
sample = df[FEATURES].iloc[X_test.index[0]].to_dict()
print('Applicant we are asking about:')
for k, v in sample.items():
    print(f'  {k}: {v}')

Applicant we are asking about:
  credit_score: 726
  annual_income: 75835.35
  loan_amount: 9235.83
  num_defaults: 0
  employment_years: 2.8
  age: 26
  loan_purpose: personal


In [ ]:
# TODO: Build a naive prompt that presents the applicant's features to the LLM
# and just asks "Should this loan be approved?"
# Call client.chat.completions.create with model='gpt-4o-mini', temperature=0
# Store the raw text in a variable called `raw_text` and print it

naive_prompt = None  # replace with your prompt string

# YOUR CODE HERE: call the API and print the raw response

raw_text = None  # replace with actual response text


---
## Section 2 — Forcing a Number

OK — let's be more explicit. Tell the model to answer with **only** 0 or 1.
Surely that will work?

We'll run 20 applicants and collect the raw strings the model returns.

In [ ]:
# Now try to use it the way we used Lab 1 models:
try:
    label = int(raw_text)
    print(f'Got label: {label}')
except (ValueError, TypeError) as e:
    print(f'CRASH: {e}')
    print()
    print('This is not a number. This is a paragraph.')
    print('XGBoost gave us a numpy array. The LLM gave us an essay.')
    print('These are fundamentally different machines.')


In [1]:
# TODO: Implement ask_for_number(applicant: dict) -> str
# Build a prompt that instructs the model: "answer with ONLY 0 or 1, no explanation"
# Call the API with max_tokens=10, temperature=0
# Return the RAW string (do not parse it yet)

def ask_for_number(applicant: dict) -> str:
    # YOUR CODE HERE
    return None


# This part is provided — run it once your function is implemented
N_PROBE = 20
test_rows = df[FEATURES].iloc[X_test.index[:N_PROBE]]

print(f'Asking the LLM about {N_PROBE} applicants...')
raw_outputs = []
for _, row in test_rows.iterrows():
    raw_outputs.append(ask_for_number(row.to_dict()))
    time.sleep(0.1)

print('\nRaw strings returned by the model:')
for i, (out, true) in enumerate(zip(raw_outputs, y_test.iloc[:N_PROBE])):
    print(f'  [{i:2d}]  repr: {repr(out):20s}   true label: {true}')

NameError: name 'df' is not defined

In [ ]:
# TODO: Visualise the variety of raw outputs
# Use Counter(raw_outputs) to count each unique string
# Plot a bar chart of the counts
# Colour the bars: green if '1' in string, red if '0' in string, orange otherwise

response_counts = Counter(raw_outputs)

# YOUR CODE HERE: bar chart of raw response counts

# TODO: Try calling int() on every raw response in a loop
# Count and print how many crash
# YOUR CODE HERE

---
## Section 3 — Writing a Robust Parser

We need a `parse_decision()` function that handles every variant the model might return:
- `'1'`, `'0'` — ideal
- `'1 '`, `' 0'` — leading/trailing whitespace
- `'1 (approve)'` — model added context anyway
- `'One'`, `'Zero'` — sometimes uses words
- something completely unexpected — default to 0 (safe fallback)

This parsing code is the **contract** between the generative model and the predictive system.

In [ ]:
# TODO: Implement parse_decision(raw: str) -> int
# Rules:
#   - If cleaned string starts with '1' -> return 1
#   - If cleaned string starts with '0' -> return 0
#   - If 'ONE', 'APPROV', or 'YES' in cleaned string -> return 1
#   - If 'ZERO', 'DENY', 'DENIED', or 'NO' in cleaned string -> return 0
#   - Otherwise -> print a warning and return 0

def parse_decision(raw: str) -> int:
    # YOUR CODE HERE
    return 0


# Test your parser on the raw outputs we already collected
parsed = [parse_decision(r) for r in raw_outputs]
print('Parser results:')
for raw, label in zip(raw_outputs, parsed):
    print(f'  {repr(raw):25s} -> {label}')


---
## Section 4 — The Proper Fix: Structured Output

The parsing approach works, but it is fragile.
OpenAI provides `response_format={"type": "json_object"}` — it forces the model to return valid JSON every time.

This is called **structured output**. Instead of hoping the model follows instructions, you constrain the output format at the API level.

In [ ]:
# TODO: Implement llm_loan_decision_v2(applicant: dict) -> int
# Instead of the fragile string prompt, use response_format={'type': 'json_object'}
# Ask the model to return {"decision": 0} or {"decision": 1}
# Parse with json.loads() and return the integer

def llm_loan_decision_v2(applicant: dict) -> int:
    # YOUR CODE HERE
    return 0


# Demonstrate: run it on a single applicant and print the raw JSON before parsing
demo = df[FEATURES].iloc[X_test.index[0]].to_dict()

# YOUR CODE HERE: call the API directly (not via llm_loan_decision_v2) and print:
# 1. The raw JSON string
# 2. The parsed dict
# 3. The final integer label

---
## Section 5 — Accuracy Check: LLM vs XGBoost

Now let's actually measure how well it does — using the same metric from Lab 1: **accuracy**.

We'll run both models on the same 50 test applicants and compare head-to-head.

In [ ]:
# Train XGBoost (same setup as Lab 1)
xgb = XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4,
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_acc   = accuracy_score(y_test, xgb_preds)
print(f'XGBoost accuracy (full test set, {len(X_test)} samples): {xgb_acc:.3f}')


In [ ]:
# TODO: Run llm_loan_decision_v2 on 50 test samples (provided below)
# Then compute accuracy for both XGBoost and the LLM on the same 50 samples

N_EVAL = 50
test_subset = df[FEATURES].iloc[X_test.index[:N_EVAL]]
y_test_subset = y_test.iloc[:N_EVAL]

print(f'Running LLM on {N_EVAL} test applicants...')
llm_preds = []
for _, row in test_subset.iterrows():
    llm_preds.append(llm_loan_decision_v2(row.to_dict()))
    time.sleep(0.1)

# YOUR CODE HERE: compute and print LLM accuracy + XGBoost accuracy on the same 50 samples

In [ ]:
# TODO: Create a 1x2 figure comparing XGBoost vs LLM accuracy
# Left: bar chart with both accuracies labelled, including the majority baseline
# Right: bar chart showing per-applicant agreement breakdown:
#   "Both correct", "Only XGBoost", "Only LLM", "Neither"

xgb_preds_sub = xgb.predict(X_test.iloc[:N_EVAL])

# YOUR CODE HERE